# YAZEL RECOVAR Integration Demo

This notebook demonstrates how RECOVAR can be used to filter false positive picks from PhaseNet.

- **RECOVAR**: Classifies waveforms to distinguish real earthquakes from noise using learned representations
- **YAZEL Integration**: Uses sliding windows to score PhaseNet picks and filter false positives

#### PhaseNet Configuration:
- **Overlap**: 0.90 (90% overlap between windows)
- **Stacking**: avg (average predictions across overlapping windows)
- **Model**: PhaseNet `instance` pretrained model


In [1]:
import obspy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from datetime import datetime

from yazel_integration_sliding import (
    recovar_pick_cleaner_sliding,
    load_recovar_classifier
)
import seisbench.models as sbm

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

2025-11-27 17:13:26.493529: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-27 17:13:26.526822: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-27 17:13:26.526859: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-27 17:13:26.526882: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-27 17:13:26.533417: I tensorflow/core/platform/cpu_feature_g

### Setup: Load Model and Data

In [ ]:
# Configuration
MODEL_PATH = '/mnt/data_a/ege/recovar_models/exp_instance/representation_learning_autoencoder_ensemble/instance/split0/ep19.h5'
PHASENET_THRESHOLD = 0.32
phasenet_pick_dir = f"filtered_phasenet_picks_dir_thr_{PHASENET_THRESHOLD:.2f}"

# Load catalog for ground truth
catalog_path = '/home/boxx/Public/earthquake_model_evaluations/data/SilivriPaper_2019-09-01__2019-11-30/processed_catalogs/kara74a_phase_picks.csv'
catalog = pd.read_csv(catalog_path)
catalog['p_arrival_time'] = pd.to_datetime(catalog['p_arrival_time'])
catalog = catalog[catalog['station'] == 'SLVT']  # Filter for SLVT station only

# Load PhaseNet picks metadata
phasenet_picks = pd.read_csv(f"{phasenet_pick_dir}/metadata.csv")

print(f"Loaded {len(phasenet_picks)} PhaseNet picks")
print(f"Loaded {len(catalog)} catalog picks for station SLVT")

In [3]:
# Load RECOVAR classifier
print("Loading RECOVAR classifier...")
classifier = load_recovar_classifier(MODEL_PATH)
print("Classifier loaded successfully!")

Loading RECOVAR classifier...


2025-11-27 17:13:56.879568: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 16953 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:19:00.0, compute capability: 8.6
2025-11-27 17:13:56.881072: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 15929 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:1a:00.0, compute capability: 8.6
2025-11-27 17:13:56.882388: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 16953 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:67:00.0, compute capability: 8.6
2025-11-27 17:13:56.886030: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 22212 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090

Classifier loaded successfully!


## Select Example Picks

For this demonstration, we'll select:
1. **TRUE PICKS**: PhaseNet picks that match catalog events (real earthquakes)
2. **FALSE PICKS**: PhaseNet picks without catalog events (noise/artifacts)

In [ ]:
# Load and categorize picks
files = sorted(Path(phasenet_pick_dir).glob('*.mseed'))

tp_examples = []
fp_examples = []

for file in files[:50]:  # Check first 50 files
    pick_row = phasenet_picks[phasenet_picks['filename'] == file.name]
    if pick_row.empty:
        continue
    
    try:
        stream = obspy.read(file)
        stream.merge()
        
        # Separate waveform traces from annotation traces (P, S, N probabilities)
        waveform_traces = [tr for tr in stream if not tr.stats.channel.endswith(('P', 'S', 'N'))]
        
        # Extract metadata from waveform
        station = waveform_traces[0].stats.station
        window_start = waveform_traces[0].stats.starttime.datetime
        window_end = waveform_traces[0].stats.endtime.datetime
        
        # Get PhaseNet pick time from metadata (this is the detection time, not in waveform)
        phasenet_pick = pd.to_datetime(pick_row['pick_time'].values[0], format='mixed')
        
        # Check if there's a catalog pick in this window
        catalog_picks = catalog[
            (catalog['p_arrival_time'] >= window_start) &
            (catalog['p_arrival_time'] <= window_end)
        ]
        
        example = {
            'file': file,
            'stream': stream,  # Keep full stream (waveforms + annotations)
            'station': station,
            'phasenet_pick': phasenet_pick,
            'window_start': window_start,
            'window_end': window_end,
            'catalog_pick': catalog_picks.iloc[0]['p_arrival_time'] if not catalog_picks.empty else None
        }
        
        if not catalog_picks.empty:
            tp_examples.append(example)
        else:
            fp_examples.append(example)
            
    except Exception as e:
        continue
    
    if len(tp_examples) >= 3 and len(fp_examples) >= 3:
        break

print(f"Found {len(tp_examples)} TRUE PICK examples")
print(f"Found {len(fp_examples)} FALSE PICK examples")

In [ ]:
def get_phasenet_probabilities(stream):
    """
    Get PhaseNet P-wave probability curve from saved annotations in the stream.
    
    Parameters:
    -----------
    stream : obspy.Stream
        Input waveform stream containing both waveform components and PhaseNet annotation traces
    
    Returns:
    --------
    dict with 'p_prob' (P-wave probabilities), 's_prob' (S-wave probabilities), 
    'times' (time array in seconds relative to stream start), 'sampling_rate'
    """
    p_trace = None
    s_trace = None
    
    for tr in stream:
        if tr.stats.channel.endswith('P'):
            p_trace = tr
        elif tr.stats.channel.endswith('S'):
            s_trace = tr
    
    # Get waveform traces to determine original starttime
    waveform_traces = [tr for tr in stream if not tr.stats.channel.endswith(('P', 'S', 'N'))]
    original_starttime = waveform_traces[0].stats.starttime if waveform_traces else stream[0].stats.starttime
    
    # Calculate time array relative to the original stream starttime
    sampling_rate = p_trace.stats.sampling_rate
    offset_seconds = (p_trace.stats.starttime - original_starttime)
    times = offset_seconds + np.arange(len(p_trace.data)) / sampling_rate
    
    return {
        'p_prob': p_trace.data,
        's_prob': s_trace.data,
        'times': times,
        'sampling_rate': sampling_rate
    }

print("PhaseNet probability extraction function ready!")

**Note:** PhaseNet probability arrays are saved directly in the MSEED files alongside waveform data during the picking phase.

## Visualization

### TRUE PICK Examples (Real Earthquake)

## PhaseNet Probability Extraction

Function to get PhaseNet P-wave probabilities over time using sliding windows.

In [ ]:
def plot_waveform_with_scores(example, recovar_result, phasenet_result, title_prefix=""):
    """
    Plot waveform with PhaseNet probabilities and RECOVAR scores overlay.
    """
    stream = example['stream']
    
    # Separate waveforms from annotations
    waveform_traces = [tr for tr in stream if not tr.stats.channel.endswith(('P', 'S', 'N')) or len(tr.stats.channel) > 1]
    z_trace = [tr for tr in waveform_traces if 'Z' in tr.stats.channel][0]
    n_trace = [tr for tr in waveform_traces if 'N' in tr.stats.channel or 'Y' in tr.stats.channel][0]
    e_trace = [tr for tr in waveform_traces if 'E' in tr.stats.channel or 'X' in tr.stats.channel][0]
    
    # Time arrays
    sampling_rate = z_trace.stats.sampling_rate
    times = np.arange(len(z_trace.data)) / sampling_rate
    
    # Calculate pick positions
    window_start = example['window_start']
    phasenet_pick = example['phasenet_pick']
    catalog_pick = example['catalog_pick']
    
    phasenet_pick_sec = (phasenet_pick - window_start).total_seconds()
    if catalog_pick:
        catalog_pick_sec = (catalog_pick - window_start).total_seconds()
    
    # Sliding window parameters for RECOVAR
    window_size = 3000  # 30 seconds at 100 Hz
    stride = 100  # 1 second at 100 Hz
    trim_samples = 500  # 5 seconds
    
    # Calculate window positions for RECOVAR score overlay
    n_windows = len(recovar_result['scores_array'])
    window_starts = []
    window_centers = []
    window_ends = []
    
    for i in range(n_windows):
        # start_idx is position in trimmed data
        start_idx = i * stride
        # Convert to position in original data
        start_orig = start_idx + trim_samples
        end_orig = start_orig + window_size
        center_orig = start_orig + window_size // 2
        
        window_starts.append(start_orig / sampling_rate)
        window_centers.append(center_orig / sampling_rate)
        window_ends.append(end_orig / sampling_rate)
    
    scores = recovar_result['scores_array']
    
    print(f"RECOVAR window coverage: {window_starts[0]:.1f}s to {window_ends[-1]:.1f}s")
    print(f"Window centers plotted: {window_centers[0]:.1f}s to {window_centers[-1]:.1f}s")
    
    # Create figure with 5 panels
    fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
    
    # Plot waveforms
    for ax, trace, comp in zip(axes[:3], [e_trace, n_trace, z_trace], ['E', 'N', 'Z']):
        ax.plot(times, trace.data, 'k-', linewidth=0.5, alpha=0.7)
        ax.set_ylabel(f'{comp}\nAmplitude', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        # Mark PhaseNet pick
        ax.axvline(phasenet_pick_sec, color='blue', linestyle='--', 
                   linewidth=2, label='PhaseNet P-pick', alpha=0.7)
        
        # Mark catalog pick if exists
        if catalog_pick:
            ax.axvline(catalog_pick_sec, color='green', linestyle='--', 
                       linewidth=2, label='Catalog P-pick', alpha=0.7)
        
        if comp == 'E':
            ax.legend(loc='upper right', fontsize=9)
    
    # Plot PhaseNet P-probabilities
    ax_phasenet = axes[3]
    pn_times = phasenet_result['times']
    pn_probs = phasenet_result['p_prob']
    
    ax_phasenet.plot(pn_times, pn_probs, 'b-', linewidth=1.5, alpha=0.8, label='PhaseNet P-probability')
    ax_phasenet.fill_between(pn_times, 0, pn_probs, color='blue', alpha=0.2)
    ax_phasenet.axhline(0.3, color='gray', linestyle=':', linewidth=1, label='Threshold (0.3)', alpha=0.6)
    ax_phasenet.axvline(phasenet_pick_sec, color='blue', linestyle='--', linewidth=2, alpha=0.5)
    
    ax_phasenet.set_ylabel('PhaseNet\nP-prob', fontsize=10)
    ax_phasenet.set_ylim(-0.05, 1.05)
    ax_phasenet.grid(True, alpha=0.3)
    ax_phasenet.legend(loc='upper right', fontsize=9)
    
    # Plot RECOVAR scores with window spans
    ax_score = axes[4]
    
    # Color-code by score value
    colors = plt.cm.RdYlGn(scores)  # Red (low) to Green (high)
    
    # Plot horizontal bars showing window coverage
    for i in range(len(window_centers)):
        # Add semi-transparent bar showing window extent
        ax_score.axvspan(window_starts[i], window_ends[i], 
                        alpha=0.1, color=colors[i], zorder=1)
        # Plot point at center
        ax_score.scatter(window_centers[i], scores[i], c=[colors[i]], 
                        s=30, alpha=0.7, edgecolors='black', linewidths=0.5, zorder=3)
    
    # Connect centers with line
    ax_score.plot(window_centers, scores, 'k-', linewidth=1, alpha=0.3, zorder=2)
    
    ax_score.axhline(recovar_result['mean_score'], color='orange', linestyle='-', 
                     linewidth=2, label=f"Mean: {recovar_result['mean_score']:.3f}", alpha=0.7)
    ax_score.axhline(recovar_result['max_score'], color='red', linestyle='--', 
                     linewidth=2, label=f"Max: {recovar_result['max_score']:.3f}", alpha=0.7)
    
    ax_score.set_ylabel('RECOVAR\nScore', fontsize=10)
    ax_score.set_xlabel('Time (seconds)', fontsize=10)
    ax_score.set_ylim(-0.05, 1.05)
    ax_score.grid(True, alpha=0.3)
    ax_score.legend(loc='upper right', fontsize=9)
    
    # Title
    status = "TRUE PICK" if catalog_pick else "FALSE PICK"
    fig.suptitle(f"{title_prefix}{status}: Station {example['station']} - "
                 f"PhaseNet Max: {np.max(pn_probs):.3f}, RECOVAR Mean: {recovar_result['mean_score']:.3f}",
                 fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Plot TRUE PICK examples
for i in range(2):
    if tp_examples:
        print(f"Processing TRUE PICK example {i+1}...")
        tp_recovar = recovar_pick_cleaner_sliding(tp_examples[i]['stream'], classifier)
        tp_phasenet = get_phasenet_probabilities(tp_examples[i]['stream'])
        plot_waveform_with_scores(tp_examples[i], tp_recovar, tp_phasenet, f"Example {i+1} - ")

### FALSE PICK Examples (Noise/Artifact)

In [ ]:
# Plot FALSE PICK examples
for i in range(1, 4):
    if fp_examples:
        print(f"Processing FALSE PICK example {i}...")
        fp_recovar = recovar_pick_cleaner_sliding(fp_examples[i]['stream'], classifier)
        fp_phasenet = get_phasenet_probabilities(fp_examples[i]['stream'])
        plot_waveform_with_scores(fp_examples[i], fp_recovar, fp_phasenet, f"Example {i+2} - ")

## Comparison: TRUE PICKS vs FALSE PICKS

Let's process multiple examples and compare the score distributions.

In [ ]:
# Process all examples
tp_results = []
fp_results = []

print("Processing TRUE PICKS...")
for example in tp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    tp_results.append(result)

print("Processing FALSE PICKS...")
for example in fp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    fp_results.append(result)

# Extract scores
tp_mean_scores = [r['mean_score'] for r in tp_results]
tp_max_scores = [r['max_score'] for r in tp_results]
fp_mean_scores = [r['mean_score'] for r in fp_results]
fp_max_scores = [r['max_score'] for r in fp_results]

print(f"\nProcessed {len(tp_results)} TRUE PICKS and {len(fp_results)} FALSE PICKS")

In [12]:
import run_yazel_batch_sliding

Loading classifier...
Loading all waveforms...
Loaded 2214 waveforms
Processing with sliding windows in batches of 256...
Creating comparison data...

=== CATALOG STATISTICS ===

Analyzed stations: 1
Total catalog P-picks for these stations: 534
PhaseNet detected (TP): 462 (86.5%)
PhaseNet missed (FN): 72 (13.5%)

=== DETECTION STATISTICS ===

Total windows: 2214
Both (catalog + PhaseNet): 497
PhaseNet only: 1717
Catalog only: 0

=== MODEL SCORES (MEAN): BOTH (TRUE POSITIVES) ===
Count: 497
Mean: 0.236
Std: 0.120
Min: 0.019
Max: 0.688

=== MODEL SCORES (MEAN): PHASENET ONLY (FALSE POSITIVES) ===
Count: 1717
Mean: 0.066
Std: 0.071
Min: 0.002
Max: 0.612

=== MODEL SCORES (MAX): BOTH (TRUE POSITIVES) ===
Count: 497
Mean: 0.379
Std: 0.170
Min: 0.030
Max: 0.961

=== MODEL SCORES (MAX): PHASENET ONLY (FALSE POSITIVES) ===
Count: 1717
Mean: 0.115
Std: 0.115
Min: 0.005
Max: 0.946

=== FINDING BEST THRESHOLD (MEAN SCORES) ===
Best threshold: 0.131373
Best F1 score: 0.729

=== FINAL PERFORMANCE 

In [ ]:
# Load the saved results from batch processing
comparison_data = pd.read_csv('SLVT_pick_comparison_sliding.csv')

# Reconstruct detection status for filtering
comparison_data['has_catalog'] = ~comparison_data['catalog_pick'].isna()

# Define threshold range to test
thresholds = np.arange(0.05, 0.21, 0.01)

# Storage for results
mean_score_results = {'thresholds': [], 'filtered_fps': [], 'missed_tps': [], 'f1_scores': []}
max_score_results = {'thresholds': [], 'filtered_fps': [], 'missed_tps': [], 'f1_scores': []}

# Calculate metrics for each threshold
for thr in thresholds:
    # Mean score filtering
    tp_mean = np.sum((comparison_data['has_catalog']) & (comparison_data['mean_score'] >= thr))
    fp_mean = np.sum((~comparison_data['has_catalog']) & (comparison_data['mean_score'] >= thr))
    fn_mean = np.sum((comparison_data['has_catalog']) & (comparison_data['mean_score'] < thr))
    tn_mean = np.sum((~comparison_data['has_catalog']) & (comparison_data['mean_score'] < thr))
    
    precision_mean = tp_mean / (tp_mean + fp_mean) if (tp_mean + fp_mean) > 0 else 0
    recall_mean = tp_mean / (tp_mean + fn_mean) if (tp_mean + fn_mean) > 0 else 0
    f1_mean = 2 * precision_mean * recall_mean / (precision_mean + recall_mean) if (precision_mean + recall_mean) > 0 else 0
    
    mean_score_results['thresholds'].append(thr)
    mean_score_results['filtered_fps'].append(tn_mean)
    mean_score_results['missed_tps'].append(fn_mean)
    mean_score_results['f1_scores'].append(f1_mean)
    
    # Max score filtering
    tp_max = np.sum((comparison_data['has_catalog']) & (comparison_data['max_score'] >= thr))
    fp_max = np.sum((~comparison_data['has_catalog']) & (comparison_data['max_score'] >= thr))
    fn_max = np.sum((comparison_data['has_catalog']) & (comparison_data['max_score'] < thr))
    tn_max = np.sum((~comparison_data['has_catalog']) & (comparison_data['max_score'] < thr))
    
    precision_max = tp_max / (tp_max + fp_max) if (tp_max + fp_max) > 0 else 0
    recall_max = tp_max / (tp_max + fn_max) if (tp_max + fn_max) > 0 else 0
    f1_max = 2 * precision_max * recall_max / (precision_max + recall_max) if (precision_max + recall_max) > 0 else 0
    
    max_score_results['thresholds'].append(thr)
    max_score_results['filtered_fps'].append(tn_max)
    max_score_results['missed_tps'].append(fn_max)
    max_score_results['f1_scores'].append(f1_max)

# Create the threshold analysis plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Mean Score - Count plot
ax1 = axes[0, 0]
ax1_twin = ax1.twinx()

line1 = ax1.plot(mean_score_results['thresholds'], mean_score_results['filtered_fps'], 
                 'g-', linewidth=2, marker='o', markersize=4, label='Filtered FALSE PICKS (TN)')
line2 = ax1_twin.plot(mean_score_results['thresholds'], mean_score_results['missed_tps'], 
                      'r-', linewidth=2, marker='s', markersize=4, label='Missed TRUE PICKS (FN)')

ax1.set_xlabel('RECOVAR Threshold (Mean Score)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Filtered FALSE PICKS (TN)', fontsize=11, color='g')
ax1_twin.set_ylabel('Missed TRUE PICKS (FN)', fontsize=11, color='r')
ax1.tick_params(axis='y', labelcolor='g')
ax1_twin.tick_params(axis='y', labelcolor='r')
ax1.grid(True, alpha=0.3)
ax1.set_title('RECOVAR Mean Score Threshold Analysis', fontsize=13, fontweight='bold')

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right', fontsize=10)

# Mean Score - F1 plot
ax2 = axes[0, 1]
ax2.plot(mean_score_results['thresholds'], mean_score_results['f1_scores'], 
         'b-', linewidth=2, marker='D', markersize=4)
best_f1_idx_mean = np.argmax(mean_score_results['f1_scores'])
best_thr_mean = mean_score_results['thresholds'][best_f1_idx_mean]
best_f1_mean = mean_score_results['f1_scores'][best_f1_idx_mean]
ax2.axvline(best_thr_mean, color='red', linestyle='--', linewidth=2, alpha=0.7,
            label=f'Best: {best_thr_mean:.2f} (F1={best_f1_mean:.3f})')
ax2.set_xlabel('RECOVAR Threshold (Mean Score)', fontsize=12, fontweight='bold')
ax2.set_ylabel('F1 Score', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_title('F1 Score vs Threshold (Mean)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)

# Max Score - Count plot
ax3 = axes[1, 0]
ax3_twin = ax3.twinx()

line3 = ax3.plot(max_score_results['thresholds'], max_score_results['filtered_fps'], 
                 'g-', linewidth=2, marker='o', markersize=4, label='Filtered FALSE PICKS (TN)')
line4 = ax3_twin.plot(max_score_results['thresholds'], max_score_results['missed_tps'], 
                      'r-', linewidth=2, marker='s', markersize=4, label='Missed TRUE PICKS (FN)')

ax3.set_xlabel('RECOVAR Threshold (Max Score)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Filtered FALSE PICKS (TN)', fontsize=11, color='g')
ax3_twin.set_ylabel('Missed TRUE PICKS (FN)', fontsize=11, color='r')
ax3.tick_params(axis='y', labelcolor='g')
ax3_twin.tick_params(axis='y', labelcolor='r')
ax3.grid(True, alpha=0.3)
ax3.set_title('RECOVAR Max Score Threshold Analysis', fontsize=13, fontweight='bold')

lines = line3 + line4
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, loc='center right', fontsize=10)

# Max Score - F1 plot
ax4 = axes[1, 1]
ax4.plot(max_score_results['thresholds'], max_score_results['f1_scores'], 
         'b-', linewidth=2, marker='D', markersize=4)
best_f1_idx_max = np.argmax(max_score_results['f1_scores'])
best_thr_max = max_score_results['thresholds'][best_f1_idx_max]
best_f1_max = max_score_results['f1_scores'][best_f1_idx_max]
ax4.axvline(best_thr_max, color='red', linestyle='--', linewidth=2, alpha=0.7,
            label=f'Best: {best_thr_max:.2f} (F1={best_f1_max:.3f})')
ax4.set_xlabel('RECOVAR Threshold (Max Score)', fontsize=12, fontweight='bold')
ax4.set_ylabel('F1 Score', fontsize=11)
ax4.grid(True, alpha=0.3)
ax4.set_title('F1 Score vs Threshold (Max)', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nMean Score - Best threshold: {best_thr_mean:.2f}")
print(f"  Filters {mean_score_results['filtered_fps'][best_f1_idx_mean]} FALSE PICKS")
print(f"  Misses {mean_score_results['missed_tps'][best_f1_idx_mean]} TRUE PICKS")
print(f"  F1 Score: {best_f1_mean:.3f}")

print(f"\nMax Score - Best threshold: {best_thr_max:.2f}")
print(f"  Filters {max_score_results['filtered_fps'][best_f1_idx_max]} FALSE PICKS")
print(f"  Misses {max_score_results['missed_tps'][best_f1_idx_max]} TRUE PICKS")
print(f"  F1 Score: {best_f1_max:.3f}")

## RECOVAR Filtering Examples

Demonstrating RECOVAR's filtering performance with threshold = 0.07 (max score):
- **TRUE PICK + Kept**: Real earthquake that RECOVAR correctly kept
- **TRUE PICK + Filtered**: Real earthquake that RECOVAR incorrectly rejected  
- **FALSE PICK + Filtered**: Noise/artifact that RECOVAR correctly rejected
- **FALSE PICK + Kept**: Noise/artifact that RECOVAR incorrectly kept

In [ ]:
def plot_side_by_side_comparison(example1, result1, phasenet1, example2, result2, phasenet2, 
                                  threshold=0.07, score_type='max'):
    """
    Plot two examples side by side for comparison.
    """
    fig, axes = plt.subplots(5, 2, figsize=(18, 12), sharex='col')
    
    for col, (example, recovar_result, phasenet_result) in enumerate([(example1, result1, phasenet1), 
                                                                        (example2, result2, phasenet2)]):
        stream = example['stream']
        
        # Separate waveforms from annotations
        waveform_traces = [tr for tr in stream if not tr.stats.channel.endswith(('P', 'S', 'N')) or len(tr.stats.channel) > 1]
        z_trace = [tr for tr in waveform_traces if 'Z' in tr.stats.channel][0]
        n_trace = [tr for tr in waveform_traces if 'N' in tr.stats.channel or 'Y' in tr.stats.channel][0]
        e_trace = [tr for tr in waveform_traces if 'E' in tr.stats.channel or 'X' in tr.stats.channel][0]
        
        # Time arrays
        sampling_rate = z_trace.stats.sampling_rate
        times = np.arange(len(z_trace.data)) / sampling_rate
        
        # Calculate pick positions
        window_start = example['window_start']
        phasenet_pick = example['phasenet_pick']
        catalog_pick = example['catalog_pick']
        
        phasenet_pick_sec = (phasenet_pick - window_start).total_seconds()
        if catalog_pick:
            catalog_pick_sec = (catalog_pick - window_start).total_seconds()
        
        # Determine filtering decision
        score_value = recovar_result['max_score'] if score_type == 'max' else recovar_result['mean_score']
        is_kept = score_value >= threshold
        is_true_pick = catalog_pick is not None
        
        # Sliding window parameters
        window_size = 3000
        stride = 100
        trim_samples = 500
        
        # Calculate window centers
        n_windows = len(recovar_result['scores_array'])
        window_centers = [(i * stride + trim_samples + window_size // 2) / sampling_rate 
                         for i in range(n_windows)]
        scores = recovar_result['scores_array']
        
        # Plot waveforms
        for row, (trace, comp) in enumerate(zip([e_trace, n_trace, z_trace], ['E', 'N', 'Z'])):
            ax = axes[row, col]
            ax.plot(times, trace.data, 'k-', linewidth=0.5, alpha=0.7)
            ax.set_ylabel(f'{comp}', fontsize=10)
            ax.grid(True, alpha=0.3)
            
            # Mark picks
            ax.axvline(phasenet_pick_sec, color='blue', linestyle='--', 
                      linewidth=2, alpha=0.7)
            if catalog_pick:
                ax.axvline(catalog_pick_sec, color='green', linestyle='--', 
                          linewidth=2, alpha=0.7)
        
        # Plot PhaseNet probabilities
        ax_phasenet = axes[3, col]
        pn_times = phasenet_result['times']
        pn_probs = phasenet_result['p_prob']
        
        ax_phasenet.plot(pn_times, pn_probs, 'b-', linewidth=1.5, alpha=0.8)
        ax_phasenet.fill_between(pn_times, 0, pn_probs, color='blue', alpha=0.2)
        ax_phasenet.axhline(0.3, color='gray', linestyle=':', linewidth=1, alpha=0.6)
        ax_phasenet.axvline(phasenet_pick_sec, color='blue', linestyle='--', linewidth=2, alpha=0.5)
        
        ax_phasenet.set_ylabel('PhaseNet\nP-prob', fontsize=10)
        ax_phasenet.set_ylim(-0.05, 1.05)
        ax_phasenet.grid(True, alpha=0.3)
        
        # Plot RECOVAR scores
        ax_score = axes[4, col]
        colors = plt.cm.RdYlGn(scores)
        
        for i in range(len(window_centers)):
            ax_score.scatter(window_centers[i], scores[i], c=[colors[i]], 
                           s=30, alpha=0.7, edgecolors='black', linewidths=0.5)
        
        ax_score.plot(window_centers, scores, 'k-', linewidth=1, alpha=0.3)
        ax_score.axhline(threshold, color='purple', linestyle='-.', linewidth=2, 
                        label=f'Threshold: {threshold:.2f}', alpha=0.8)
        ax_score.axhline(score_value, color='red', linestyle='--', linewidth=2,
                        label=f'{score_type.title()}: {score_value:.3f}', alpha=0.7)
        
        ax_score.set_ylabel('RECOVAR\nScore', fontsize=10)
        ax_score.set_xlabel('Time (seconds)', fontsize=10)
        ax_score.set_ylim(-0.05, 1.05)
        ax_score.grid(True, alpha=0.3)
        ax_score.legend(loc='upper right', fontsize=8)
        
        # Column title
        pick_type = "TRUE PICK" if is_true_pick else "FALSE PICK"
        decision = "KEPT" if is_kept else "FILTERED"
        decision_color = 'green' if (is_true_pick and is_kept) or (not is_true_pick and not is_kept) else 'red'
        
        axes[0, col].set_title(f"{pick_type} - {decision}\nStation: {example['station']}", 
                              fontsize=12, fontweight='bold', color=decision_color, pad=10)
    
    plt.tight_layout()
    plt.show()

# Find examples of each category using the threshold of 0.07
RECOVAR_THRESHOLD = 0.07

# Categorize examples
tp_kept = []  # TRUE PICK kept by RECOVAR
tp_filtered = []  # TRUE PICK filtered by RECOVAR
fp_kept = []  # FALSE PICK kept by RECOVAR
fp_filtered = []  # FALSE PICK filtered by RECOVAR

for example in tp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    example['recovar_result'] = result
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if result['max_score'] >= RECOVAR_THRESHOLD:
        tp_kept.append(example)
    else:
        tp_filtered.append(example)

for example in fp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    example['recovar_result'] = result
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if result['max_score'] >= RECOVAR_THRESHOLD:
        fp_kept.append(example)
    else:
        fp_filtered.append(example)

print(f"TRUE PICKS kept by RECOVAR: {len(tp_kept)}")
print(f"TRUE PICKS filtered by RECOVAR: {len(tp_filtered)}")
print(f"FALSE PICKS kept by RECOVAR: {len(fp_kept)}")
print(f"FALSE PICKS filtered by RECOVAR: {len(fp_filtered)}")

# Plot comparisons
if tp_kept and fp_filtered:
    print("\n--- Comparison 1: Correct Decisions ---")
    plot_side_by_side_comparison(tp_kept[0], tp_kept[0]['recovar_result'], tp_kept[0]['phasenet_result'],
                                fp_filtered[0], fp_filtered[0]['recovar_result'], fp_filtered[0]['phasenet_result'],
                                threshold=RECOVAR_THRESHOLD)

if tp_filtered and fp_kept:
    print("\n--- Comparison 2: Incorrect Decisions ---")
    plot_side_by_side_comparison(tp_filtered[0] if tp_filtered else tp_kept[0], 
                                tp_filtered[0]['recovar_result'] if tp_filtered else tp_kept[0]['recovar_result'],
                                tp_filtered[0]['phasenet_result'] if tp_filtered else tp_kept[0]['phasenet_result'],
                                fp_kept[0], fp_kept[0]['recovar_result'], fp_kept[0]['phasenet_result'],
                                threshold=RECOVAR_THRESHOLD)

## Summary
- A well-chosen RECOVAR threshold can filter many false positives while retaining most true positives

For batch processing and full evaluation, see `run_yazel_batch_sliding.py`